# 06 — Backend (FastAPI)

`backend/app/main.py` wires everything together: model loading at startup (once, cached in `app.state.model_registry`), migrations, CORS, rate limiting, request timing.

In [ ]:
import os
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from wherever this notebook actually lives to find the real
    project root (the folder containing both backend/app/ and data/), so every
    relative path used below resolves correctly regardless of which folder
    this notebook is opened from."""
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "app").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the Baseera project root (a folder containing both "
        "backend/app/ and data/) above this notebook's location."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
print("Project root:", PROJECT_ROOT)


In [ ]:
import inspect
from app import main as backend_main

print(inspect.getsource(backend_main.lifespan))


**Endpoints** (`app/api/v1/`): `/sentiment/predict`, `/predict-batch`, `/pipeline` (all 3 tasks together), `/explain` (SHAP), `/upload-file` (batch CSV/Excel), `/analyses` (history), plus read-only analytics endpoints backing the dashboard. Full detail: `API_DOCUMENTATION.md`.

**Idempotency**: `POST /predict` accepts an `Idempotency-Key` header -- a client retry after a network timeout replays the already-saved result instead of creating a duplicate history row.

**Layered architecture**: `api/v1/endpoints` (HTTP) -> `services` (business logic, no FastAPI imports) -> `ml` (pure ML/data code) + `repositories` (cached data access).

## Database & persistence layer

Optional relational persistence (sentiment-analysis history, feedback, durable batch-upload records) via SQLAlchemy + Alembic, entirely opt-in via `DATABASE_URL` -- see `DATABASE_SETUP.md`. The app runs fully without it (predictions still work, uploads persist to local JSON files for 7 days instead).

In [ ]:
import inspect
from app.db import models as db_models

print([name for name, obj in inspect.getmembers(db_models) if inspect.isclass(obj) and obj.__module__ == db_models.__name__])
